# Batch Scoring - Manufacturing Predictive Maintenance

Scores every machine-day with the trained RandomForest model to produce
failure-risk predictions and a per-machine risk summary.

Reads from: gold_ml_features and the saved model at Files/models/maintenance_failure_rf
Writes to: gold_ml_predictions, gold_ml_summary

In [ ]:
from pyspark.sql.functions import (
    col, lit, current_timestamp, when, avg, count,
    sum as spark_sum, round as spark_round, isnan, udf
)
from pyspark.sql.types import DoubleType
from pyspark.ml.feature import VectorAssembler
from pyspark.ml.classification import RandomForestClassificationModel

# Load the model saved by the training notebook
model = RandomForestClassificationModel.load('Files/models/maintenance_failure_rf')
df = spark.read.table('gold_ml_features')
print(f'Scoring {df.count()} feature rows')

In [ ]:
# Clean nulls / NaN and rebuild the exact feature set used during training
for c, dtype in df.dtypes:
    if dtype in ('double', 'float'):
        df = df.withColumn(c, when(col(c).isNull() | isnan(col(c)), lit(0.0)).otherwise(col(c)))
    elif dtype in ('int', 'bigint', 'long'):
        df = df.withColumn(c, when(col(c).isNull(), lit(0)).otherwise(col(c)))

exclude = ('had_failure', 'failure_event_count', 'total_failure_downtime', 'total_failure_defects')
feature_cols = [
    c for c, dtype in df.dtypes
    if dtype in ('double', 'float', 'int', 'bigint', 'long') and c not in exclude
]
print(f'Features ({len(feature_cols)}): {feature_cols}')

assembler = VectorAssembler(inputCols=feature_cols, outputCol='features', handleInvalid='keep')
model_df = assembler.transform(df)

In [ ]:
# Score every row and derive failure probability + risk level
scored = model.transform(model_df)

extract_prob = udf(lambda v: float(v[1]) if v is not None and len(v) > 1 else 0.0, DoubleType())

predictions = (
    scored
    .withColumn('failure_probability', spark_round(extract_prob(col('probability')), 4))
    .withColumn('predicted_failure', col('prediction').cast('int'))
    .withColumn('risk_level',
        when(col('failure_probability') > 0.8, 'critical')
        .when(col('failure_probability') > 0.6, 'high')
        .when(col('failure_probability') > 0.4, 'medium')
        .otherwise('low')
    )
    .withColumn('scored_at', current_timestamp())
    .select(
        'machine_id', 'production_line', 'sensor_date',
        'avg_temp', 'avg_pressure', 'avg_vibration', 'avg_humidity',
        'temp_range', 'pressure_range', 'vibration_range',
        'anomaly_ratio', 'equipment_age_days',
        'had_failure', 'predicted_failure',
        'failure_probability', 'risk_level',
        'scored_at'
    )
)

predictions.write.mode('overwrite').option('overwriteSchema', 'true').format('delta').saveAsTable('gold_ml_predictions')
print(f'Predictions written: {predictions.count()} rows')
predictions.groupBy('risk_level').count().orderBy('count', ascending=False).show()

In [ ]:
# Per-machine risk summary
summary = (
    predictions
    .groupBy('machine_id', 'production_line')
    .agg(
        spark_round(avg('failure_probability'), 4).alias('avg_failure_risk'),
        spark_sum('predicted_failure').alias('predicted_failure_days'),
        count('*').alias('total_days'),
        spark_round(avg('avg_temp'), 2).alias('avg_daily_temp'),
        spark_round(avg('avg_pressure'), 2).alias('avg_daily_pressure'),
        spark_round(avg('avg_vibration'), 4).alias('avg_daily_vibration'),
    )
    .withColumn('failure_rate', spark_round(col('predicted_failure_days') / col('total_days') * 100, 1))
    .withColumn('overall_risk',
        when(col('avg_failure_risk') > 0.6, 'high')
        .when(col('avg_failure_risk') > 0.3, 'medium')
        .otherwise('low')
    )
    .withColumn('summary_timestamp', current_timestamp())
    .orderBy(col('avg_failure_risk').desc())
)

summary.write.mode('overwrite').option('overwriteSchema', 'true').format('delta').saveAsTable('gold_ml_summary')
print(f'Machine risk summary: {summary.count()} machines')
summary.show(10, truncate=False)

In [ ]:
# Optimize
spark.sql('OPTIMIZE gold_ml_predictions')
spark.sql('OPTIMIZE gold_ml_summary')
print('\nAll Gold ML tables optimized')

print('\n=== Scoring Summary ===')
print(f'Total predictions: {predictions.count():,}')
print(f'Machines scored: {summary.count()}')
high_risk = summary.filter(col('overall_risk') == 'high').count()
print(f'High-risk machines: {high_risk}')
print('\nGold tables created:')
print('  - gold_ml_features')
print('  - gold_ml_model_metrics')
print('  - gold_ml_feature_importance')
print('  - gold_ml_predictions')
print('  - gold_ml_summary')